# 02. SegResNet 3D Inference & Lesion Visualization (ISLES-2022)

**Mục tiêu:** Nạp weights tốt nhất từ đợt train SegResNet 2 epochs (`best_metric_model.pth`), chạy sliding-window inference trên **Validation Case 11**, tự động tìm lát cắt có ổ nhồi máu lớn nhất và trực quan hóa so sánh đa khung hình.

In [ ]:
import os
import sys
from pathlib import Path

# Thêm thư mục gốc dự án vào sys.path
PROJECT_ROOT = Path(os.path.abspath("")).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from monai.inferers import sliding_window_inference

from src.dataset.dataloader import build_kfold_dataloaders
from src.metrics.dice import compute_dice
from src.models.builder import get_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# 1. Khởi tạo mô hình SegResNet và nạp checkpoint best_metric_model.pth
ckpt_path = PROJECT_ROOT / "experiments" / "runs" / "segresnet_fold0_20260922_213311" / "best_metric_model.pth"
print(f"Loading checkpoint from: {ckpt_path}")

model = get_model("segresnet", in_channels=3, out_channels=1).to(device)
checkpoint = torch.load(ckpt_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

best_dice = checkpoint.get("best_dice", checkpoint.get("val_dice", "N/A"))
print(f"Model SegResNet loaded successfully! Best Validation Dice: {best_dice}")

In [ ]:
# 2. Nạp Validation Case 11 từ Fold 0
_, val_loader = build_kfold_dataloaders(
    data_dir=PROJECT_ROOT / "data" / "raw" / "ISLES-2022",
    fold=0,
    n_splits=5,
    batch_size=1,
)

val_case = None
for idx, batch in enumerate(val_loader):
    if idx == 10:  # Case 11 tương ứng index 10 (1-based index)
        val_case = batch
        break

images = val_case["image"].to(device)  # shape [1, 3, H, W, D]
labels = val_case["mask"].to(device)   # shape [1, 1, H, W, D]
print(f"Case 11 loaded - Images shape: {images.shape}, Masks shape: {labels.shape}")

In [ ]:
# 3. Chạy Sliding Window Inference trên toàn bộ thể tích 3D não
with torch.no_grad():
    outputs = sliding_window_inference(
        inputs=images,
        roi_size=(96, 96, 32),
        sw_batch_size=1,
        predictor=model,
        overlap=0.25,
        mode="gaussian",
    )

preds = (torch.sigmoid(outputs) > 0.5).float()
case_dice = compute_dice(preds[0, 0], labels[0, 0])
print(f"Inference hoàn tất! Case 11 3D Dice Score: {case_dice:.4f}")

In [ ]:
# 4. Tìm lát cắt và trực quan hóa kết quả
dwi_vol = images[0, 0].cpu().numpy()
gt_vol = labels[0, 0].cpu().numpy()
pred_vol = preds[0, 0].cpu().numpy()

# Logic tìm lát cắt: Quét dọc theo trục Z (trục axial), tính tổng diện tích tổn thương thực tế (Ground Truth)
# và chọn ra lát cắt có số lượng voxel tổn thương lớn nhất để hiển thị trực quan rõ nét nhất
z_areas = [gt_vol[:, :, z].sum() for z in range(gt_vol.shape[2])]
best_z = int(np.argmax(z_areas))
max_lesion_voxels = z_areas[best_z]
print(f"Lát cắt tối ưu được chọn: Z = {best_z} (Chứa {int(max_lesion_voxels)} voxels nhãn tổn thương)")

# Xoay ảnh 90 độ ngược chiều kim đồng hồ để hiển thị hướng giải phẫu chuẩn
dwi_slice = np.rot90(dwi_vol[:, :, best_z])
gt_slice = np.rot90(gt_vol[:, :, best_z])
pred_slice = np.rot90(pred_vol[:, :, best_z])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (1) Ảnh DWI gốc
im0 = axes[0].imshow(dwi_slice, cmap="gray")
axes[0].set_title(f"(1) Raw DWI (Slice Z={best_z})", fontsize=13, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# (2) Ground Truth Mask phủ trên nền DWI
axes[1].imshow(dwi_slice, cmap="gray")
axes[1].imshow(np.ma.masked_where(gt_slice == 0, gt_slice), cmap="autumn", alpha=0.75)
axes[1].set_title(f"(2) Ground Truth Mask ({int(max_lesion_voxels)} voxels)", fontsize=13, fontweight="bold", color="darkred")
axes[1].axis("off")

# (3) Predicted Mask phủ trên nền DWI
axes[2].imshow(dwi_slice, cmap="gray")
axes[2].imshow(np.ma.masked_where(pred_slice == 0, pred_slice), cmap="cool", alpha=0.75)
axes[2].set_title(f"(3) Predicted Mask (Dice: {case_dice:.4f})", fontsize=13, fontweight="bold", color="navy")
axes[2].axis("off")

plt.suptitle(f"ISLES-2022 Validation Case 11 - SegResNet 2 Epochs Slice Comparison (Z={best_z})", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Dọn dẹp và giải phóng bộ nhớ đệm VRAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM cache cleared successfully.")